In [1]:
#!/usr/bin/env python3
"""
Spark Structured Streaming with PostgreSQL
FINAL VERSION - With proper cleanup
"""

import os
import time
import logging
import random
import csv
import shutil
import subprocess
import sys
import signal
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# ============================================================================
# CONFIGURATION
# ============================================================================

DB_CONFIG = {
    "url": "jdbc:postgresql://localhost:5432/Data_engineering_practice",
    "user": "postgres",
    "password": "xxxxxx",  # 🔴 CHANGE THIS
    "driver": "org.postgresql.Driver"
}

BASE_DIR = "/home/odinsbeard/Data_engineering_Journey/week6_spark"
JAR_PATH = f"{BASE_DIR}/jars/postgresql-42.7.1.jar"
CHECKPOINT_DIR = f"{BASE_DIR}/checkpoints"
STREAMING_DIR = f"{BASE_DIR}/data/streaming"

# Clean up old checkpoints
if os.path.exists(CHECKPOINT_DIR):
    shutil.rmtree(CHECKPOINT_DIR)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(f"{STREAMING_DIR}/input", exist_ok=True)
os.makedirs(f"{STREAMING_DIR}/output", exist_ok=True)

# ============================================================================
# SETUP LOGGING
# ============================================================================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# ============================================================================
# CREATE SEPARATE SIMULATOR SCRIPT
# ============================================================================
def create_simulator_script():
    """Create a separate Python script for the simulator"""
    simulator_path = f"{BASE_DIR}/scripts/order_simulator.py"
    
    script_content = '''#!/usr/bin/env python3
import csv
import random
import time
import os
import sys
from datetime import datetime

STREAMING_DIR = r"{streaming_dir}"

# Create a flag file to know when to stop
stop_flag = os.path.join(STREAMING_DIR, "SIMULATOR_RUNNING")

# Create stop flag
with open(stop_flag, 'w') as f:
    f.write("running")

customers = list(range(1, 101))
products = list(range(1, 51))

batch_num = 1
try:
    while os.path.exists(stop_flag):
        try:
            filename = os.path.join(STREAMING_DIR, "input", f"orders_{{datetime.now().strftime('%Y%m%d_%H%M%S')}}.csv")
            
            with open(filename, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(["order_id", "customer_id", "product_id", "quantity", "unit_price", "timestamp"])
                
                for _ in range(random.randint(3, 7)):
                    order_id = batch_num * 100 + random.randint(1, 100)
                    timestamp = datetime.now().isoformat()
                    unit_price = random.uniform(10, 200)
                    
                    writer.writerow([
                        order_id,
                        random.choice(customers),
                        random.choice(products),
                        random.randint(1, 5),
                        round(unit_price, 2),
                        timestamp
                    ])
            
            print(f"📁 Created batch {{batch_num}}: {{os.path.basename(filename)}}")
            batch_num += 1
            time.sleep(10)
            
        except Exception as e:
            print(f"⚠️ Simulator warning: {{e}}")
            time.sleep(5)
finally:
    # Clean up stop flag
    if os.path.exists(stop_flag):
        os.remove(stop_flag)
    print("✅ Simulator stopped cleanly")
'''
    
    with open(simulator_path, 'w') as f:
        f.write(script_content.format(streaming_dir=STREAMING_DIR))
    
    os.chmod(simulator_path, 0o755)
    logger.info(f"✅ Simulator script created at {simulator_path}")
    return simulator_path

# ============================================================================
# INITIALIZE SPARK
# ============================================================================
def init_spark():
    spark = SparkSession.builder \
        .appName("Spark Streaming with PostgreSQL") \
        .config("spark.jars", JAR_PATH) \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.streaming.checkpointLocation", CHECKPOINT_DIR) \
        .getOrCreate()
    
    spark.sparkContext.setLogLevel("WARN")
    logger.info(f"✅ Spark session created (version: {spark.version})")
    return spark

# ============================================================================
# STREAMING SOURCE
# ============================================================================
def create_file_stream(spark):
    schema = StructType([
        StructField("order_id", IntegerType()),
        StructField("customer_id", IntegerType()),
        StructField("product_id", IntegerType()),
        StructField("quantity", IntegerType()),
        StructField("unit_price", DoubleType()),
        StructField("timestamp", TimestampType())
    ])
    
    stream_df = spark.readStream \
        .schema(schema) \
        .option("header", "true") \
        .option("maxFilesPerTrigger", 1) \
        .csv(f"{STREAMING_DIR}/input")
    
    stream_df = stream_df \
        .withColumn("total_amount", col("quantity") * col("unit_price")) \
        .withColumn("year", year("timestamp")) \
        .withColumn("month", month("timestamp")) \
        .withColumn("day", dayofmonth("timestamp")) \
        .withColumn("hour", hour("timestamp"))
    
    return stream_df

# ============================================================================
# WRITE TO CONSOLE
# ============================================================================
def write_to_console(df, trigger_interval="10 seconds"):
    query = df.writeStream \
        .outputMode("append") \
        .trigger(processingTime=trigger_interval) \
        .format("console") \
        .option("truncate", "false") \
        .option("numRows", 10) \
        .start()
    return query

# ============================================================================
# WRITE TO MEMORY
# ============================================================================
def write_to_memory(df, table_name):
    query = df.writeStream \
        .outputMode("append") \
        .format("memory") \
        .queryName(table_name) \
        .option("checkpointLocation", f"{CHECKPOINT_DIR}/{table_name}") \
        .start()
    return query

# ============================================================================
# WRITE TO POSTGRESQL
# ============================================================================
def write_to_postgresql(stream_df, table_name):
    def write_batch(df, epoch_id):
        count = df.count()
        if count > 0:
            logger.info(f"📦 Writing batch {epoch_id} with {count} rows to PostgreSQL")
            df.write \
                .mode("append") \
                .format("jdbc") \
                .option("url", DB_CONFIG["url"]) \
                .option("dbtable", table_name) \
                .option("user", DB_CONFIG["user"]) \
                .option("password", DB_CONFIG["password"]) \
                .option("driver", DB_CONFIG["driver"]) \
                .option("batchsize", 10000) \
                .save()
            logger.info(f"✅ Batch {epoch_id} written to PostgreSQL")
    
    query = stream_df.writeStream \
        .trigger(processingTime="30 seconds") \
        .foreachBatch(write_batch) \
        .option("checkpointLocation", f"{CHECKPOINT_DIR}/postgres_{table_name}") \
        .start()
    return query

# ============================================================================
# QUERY MEMORY TABLE
# ============================================================================
def query_memory_table(spark, table_name):
    result = spark.sql(f"SELECT COUNT(*) as count FROM {table_name}")
    count = result.collect()[0][0]
    logger.info(f"📊 {table_name} has {count} rows")
    if count > 0:
        spark.sql(f"SELECT * FROM {table_name} ORDER BY timestamp DESC LIMIT 5").show(truncate=False)

# ============================================================================
# CLEANUP FUNCTION
# ============================================================================
def cleanup_simulator():
    """Stop the simulator cleanly"""
    stop_flag = os.path.join(STREAMING_DIR, "SIMULATOR_RUNNING")
    if os.path.exists(stop_flag):
        os.remove(stop_flag)
        logger.info("✅ Sent stop signal to simulator")
        time.sleep(2)  # Give it time to stop

# ============================================================================
# MAIN
# ============================================================================
def main():
    print("\n" + "=" * 70)
    print("🚀 SPARK STREAMING DEMONSTRATION")
    print("=" * 70)
    
    simulator_process = None
    
    try:
        # Clean up any old simulator
        cleanup_simulator()
        time.sleep(2)
        
        # Create simulator script
        simulator_path = create_simulator_script()
        
        # Start simulator in separate process
        logger.info("📦 Starting order simulator...")
        simulator_process = subprocess.Popen([sys.executable, simulator_path], 
                                             stdout=subprocess.PIPE, 
                                             stderr=subprocess.STDOUT,
                                             text=True)
        logger.info("✅ Simulator started")
        
        # Initialize Spark
        spark = init_spark()
        
        # Wait for first files
        time.sleep(5)
        
        # Create stream
        stream_df = create_file_stream(spark)
        
        # Start queries
        console_query = write_to_console(stream_df, "10 seconds")
        memory_query = write_to_memory(stream_df, "live_orders")
        pg_query = write_to_postgresql(stream_df, "streaming_orders")
        
        logger.info("\n⏱️  Streaming pipeline running for 2 minutes...")
        logger.info("   Press Ctrl+C to stop early\n")
        
        # Monitor
        for i in range(6):
            time.sleep(20)
            logger.info(f"\n🔍 Querying live_orders table (after {20*(i+1)} seconds):")
            query_memory_table(spark, "live_orders")
        
    except KeyboardInterrupt:
        logger.info("\n⚠️  Stopped by user")
    except Exception as e:
        logger.error(f"❌ Error: {e}")
    finally:
        # Stop everything
        logger.info("\n🛑 Stopping all streaming queries...")
        try:
            console_query.stop()
            memory_query.stop()
            pg_query.stop()
        except:
            pass
        
        # Stop simulator
        if simulator_process:
            cleanup_simulator()
            simulator_process.terminate()
            simulator_process.wait(timeout=5)
        
        time.sleep(3)
        spark.stop()
        logger.info("✅ Spark session stopped")
        
        # Final cleanup
        cleanup_simulator()

if __name__ == "__main__":
    main()




🚀 SPARK STREAMING DEMONSTRATION


2026-03-17 16:17:08,604 - INFO - ✅ Simulator script created at /home/odinsbeard/Data_engineering_Journey/week6_spark/scripts/order_simulator.py
2026-03-17 16:17:08,607 - INFO - 📦 Starting order simulator...
2026-03-17 16:17:08,610 - INFO - ✅ Simulator started
26/03/17 16:17:16 WARN Utils: Your hostname, odinsbeard-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/03/17 16:17:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/17 16:17:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
2026-03-17 16:17:25,812 - INFO - ✅ Spark session created (version: 3.5.0)
26/03/17 16:17:42 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/03

-------------------------------------------
Batch: 0
-------------------------------------------
+--------+-----------+----------+--------+----------+---------+------------+----+-----+---+----+
|order_id|customer_id|product_id|quantity|unit_price|timestamp|total_amount|year|month|day|hour|
+--------+-----------+----------+--------+----------+---------+------------+----+-----+---+----+
+--------+-----------+----------+--------+----------+---------+------------+----+-----+---+----+



26/03/17 16:17:56 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 10000 milliseconds, but spent 13071 milliseconds


-------------------------------------------
Batch: 1
-------------------------------------------
+--------+-----------+----------+--------+----------+---------+------------+----+-----+---+----+
|order_id|customer_id|product_id|quantity|unit_price|timestamp|total_amount|year|month|day|hour|
+--------+-----------+----------+--------+----------+---------+------------+----+-----+---+----+
+--------+-----------+----------+--------+----------+---------+------------+----+-----+---+----+



...

2026-03-17 16:19:26,679 - INFO - 
🔍 Querying live_orders table (after 100 seconds):
2026-03-17 16:19:30,144 - INFO - 📊 live_orders has 62 rows
2026-03-17 16:19:30,306 - INFO - Received command c on object id p0


-------------------------------------------
Batch: 11
-------------------------------------------
+--------+-----------+----------+--------+----------+---------+------------+----+-----+---+----+
|order_id|customer_id|product_id|quantity|unit_price|timestamp|total_amount|year|month|day|hour|
+--------+-----------+----------+--------+----------+---------+------------+----+-----+---+----+
+--------+-----------+----------+--------+----------+---------+------------+----+-----+---+----+

+--------+-----------+----------+--------+----------+--------------------------+------------+----+-----+---+----+
|order_id|customer_id|product_id|quantity|unit_price|timestamp                 |total_amount|year|month|day|hour|
+--------+-----------+----------+--------+----------+--------------------------+------------+----+-----+---+----+
|1482    |22         |48        |2       |148.16    |2026-03-14 12:14:55.08537 |296.32      |2026|3    |14 |12  |
|1448    |29         |30        |5       |39.79     |2026

2026-03-17 16:19:50,882 - INFO - 
🔍 Querying live_orders table (after 120 seconds):
2026-03-17 16:19:51,165 - INFO - 📊 live_orders has 75 rows
2026-03-17 16:19:51,330 - INFO - 
🛑 Stopping all streaming queries...


+--------+-----------+----------+--------+----------+--------------------------+------------+----+-----+---+----+
|order_id|customer_id|product_id|quantity|unit_price|timestamp                 |total_amount|year|month|day|hour|
+--------+-----------+----------+--------+----------+--------------------------+------------+----+-----+---+----+
|1482    |22         |48        |2       |148.16    |2026-03-14 12:14:55.08537 |296.32      |2026|3    |14 |12  |
|1448    |29         |30        |5       |39.79     |2026-03-14 12:14:55.085365|198.95      |2026|3    |14 |12  |
|1493    |61         |45        |3       |40.49     |2026-03-14 12:14:55.085359|121.47      |2026|3    |14 |12  |
|1492    |33         |45        |2       |151.59    |2026-03-14 12:14:55.085352|303.18      |2026|3    |14 |12  |
|1421    |78         |4         |5       |107.61    |2026-03-14 12:14:55.085344|538.05      |2026|3    |14 |12  |
+--------+-----------+----------+--------+----------+--------------------------+--------

2026-03-17 16:19:51,630 - INFO - ✅ Sent stop signal to simulator
2026-03-17 16:19:58,110 - INFO - ✅ Spark session stopped
